[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/flame/flame.ipynb)

# Cellular Flame

This example computes flames spreading from sparks through a lean mixture of fuel and air and merging into a front that breaks into cells as it grows. A premixed flame is a thin front where fuel burns, advancing into the unburnt mixture as the heat conducted ahead ignites the fuel there. Because the reaction rate grows steeply with temperature under Arrhenius's law, the front's speed depends strongly on its temperature. A bulge in the front sheds heat into a wider sector of unburnt gas and draws fuel from a wider sector as well. Where fuel diffuses faster than heat, as hydrogen does in lean air, fuel transport prevails, so the bulge burns hotter and advances faster while the creases between bulges lose more heat than incoming fuel replaces, cool down and nearly extinguish. The Lewis number, thermal diffusivity divided by fuel diffusivity, governs this balance, and below one a smooth front is unstable. Constant density leaves out the gas's thermal expansion flow across the front.

The algebra joins the plane `x+ y+` with three degenerate directions `v0 f0 h0` for the quantities volume, fuel and heat. A parcel of the mixture is modelled as a sum of those units, one unit of volume `v.dual()` plus its fuel and its heat, read off by `f & parcel` and `h & parcel`. Burning is one fixed direction among the quantities, `reaction = heat - fuel`, and chemical consumption steps each parcel along that direction while leaving its volume unchanged. The plane `energy = h + f` reads a parcel's enthalpy, the sum of its sensible heat and the chemical heat stored in its fuel, and pairs with the reaction to zero, `energy & reaction == 0`, so burning converts chemical energy into sensible heat without creating energy. A drive `-gradient ^ potentials` wedges minus a simplex gradient with a quantity's direction, giving `-gradients ^ h` for heat and `-gradients ^ f` for fuel, and separate isotropic material maps take each drive to its flux. Diagonal drives leave out Soret and Dufour cross-diffusion between temperature and concentration gradients. Because reaction leaves `energy & parcels` unchanged, enthalpy varies only because fuel and heat diffuse at different rates, concentrating behind the bulges and depleting along the creases.

In scalar reaction–diffusion partial differential equations, dimensionless temperature $T$ and fuel mass fraction $Y$ obey $\partial_t T = \nabla^2 T + \omega(T, Y)$ and $\partial_t Y = \mathrm{Le}^{-1}\nabla^2 Y - \omega(T, Y)$, whose sum cancels the Arrhenius source $\omega$ in the enthalpy balance $\partial_t(T + Y) = \nabla^2(T + Y) + (\mathrm{Le}^{-1} - 1)\nabla^2 Y$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import jax
import numpy as np

from numga import Algebra, stack
from numga.backend.jax import JaxContext
from numga.sparse import spdiag
from examples.util.animation import show_states
from numga.topology import Topology
from examples.thermodynamics.flame import render, scenarios

jax.config.update("jax_enable_x64", True)

# The plane `x+ y+` joined with three degenerate directions `v0 f0 h0` for the quantities volume, fuel and heat.
space = Algebra("x+y+")
quantities = Algebra("v0f0h0")
ga = space * quantities
mv = JaxContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
# Cell-integrated areas and quantities carry the intrinsic orientation of the five-way join.
Antiscalar = ga.gatype.antipseudoscalar()                                     # Antipseudoscalar-
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
# Spatial planes reading horizontal and vertical offsets.
Upright = Plane.derive.within(space)                                          # Vector+
Drive = ga.gatype.bivector().derive.touching(space, Algebra("f0h0"))          # Bivector+
# Horizontal and vertical directions in the plane.
directions = stack((mv.x.dual(), mv.y.dual()))                                # [2] Antivector-
# Units of the three quantities: volume, fuel and heat.
volume = mv.v.dual()                                                          # [] Antivector-
fuel = mv.f.dual()                                                            # [] Antivector-
heat = mv.h.dual()                                                            # [] Antivector-
# Enthalpy plane reading the sum of sensible heat and chemical fuel energy.
energy = mv.h + mv.f                                                          # [] Vector+

## The disc and the sparks

The mixture fills a triangulated disc of staggered, nearly equilateral rings so that no spatial direction is favoured and the thin reaction zone spans several triangles. Lengths are measured in flame thicknesses and times in flame transit times across one thickness. Joining a triangle's facing side with the three quantity directions and dividing by twice the area gives each corner's simplex gradient, which drives all diffusive transport below. Triangle areas and cell-integrated quantities keep the intrinsic orientation of the five-way join, and dividing a cell outflow by the cell's area converts it into a local rate. Three sparks of unequal radii with noise-roughened rims ignite three expanding fronts.

In [ ]:
def triangles(topology: Topology, vertices: Point) -> tuple[Antiscalar, Plane, Antiscalar]:
    """Triangle areas, corner simplex gradients, and dual vertex cell areas across the triangulated disc."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)              # [3] Antivector-[F]
    # Joining two sides with unit volume, fuel and heat gives twice the oriented triangle area.
    double_areas = (((sides[0] & sides[1]) & volume) & fuel) & heat           # Antipseudoscalar-[F]
    areas = double_areas / 2                                                  # Antipseudoscalar-[F]
    # Joining the facing side with volume, fuel and heat over twice the area gives each corner's simplex gradient.
    gradients = (volume & fuel & heat & sides) / double_areas                 # [3] Vector+[F]
    return areas, gradients, (topology.incidence[2, 0].adjugate() * (areas / 3)).sum(axis=0)


# Triangulated disc and its spatial vertex offsets.
topology, coordinates = scenarios.disc(scenarios.n_rings)
vertices = mv(Upright, coordinates).dual().field()                            # Antivector-[V]
areas, gradients, cells = triangles(topology, vertices)
# Initial parcel field: burnt sparks hold sensible heat where fuel was consumed, surrounded by unburnt fuel.
burnt_fraction = scenarios.hot_spot(coordinates)                              # Scalar+[V]
parcels = volume + heat * burnt_fraction + fuel * (1 - burnt_fraction)        # Antivector-[V]
render.draw_start(topology, vertices, scenarios.radius, energy, parcels);

On the left, the three sparks appear charred dark inside the pale unburnt mixture; on the right, the enthalpy field is uniform across the disc because sensible heat inside the sparks equals the chemical energy of the displaced fuel.

## Burning

Each unit of fuel burns at an Arrhenius rate proportional to the exponential of minus the activation temperature divided by the absolute temperature. The Zeldovich number sets the steepness of that exponential near the adiabatic flame temperature, and the prefactor normalizes the flat-flame speed to unity in the high-activation-energy limit. Over one time step a parcel's fuel decays exponentially at its local rate, and the consumed fraction steps the parcel along the reaction direction, releasing one unit of heat per unit of fuel. The Lewis number matches lean hydrogen in air, and the time step resolves rapid burning inside the hot front.

In [ ]:
# Lewis number, Zeldovich number, unburnt temperature in adiabatic-rise units, and time step.
lewis, zeldovich, unburnt_temperature = scenarios.lewis, scenarios.zeldovich, scenarios.unburnt_temperature
dt = scenarios.dt
# Activation temperature and prefactor normalizing the flat-flame speed to unity.
burnt_temperature = unburnt_temperature + 1
activation = zeldovich * burnt_temperature ** 2
prefactor = zeldovich ** 2 / (2 * lewis) * np.exp(activation / burnt_temperature)
# Reaction direction among the quantities, converting one unit of fuel into one unit of heat.
reaction = heat - fuel                                                        # [] Antivector-


def fuel_loss(parcels: Point) -> Scalar:
    """Fuel consumed in each parcel over one time step under Arrhenius decay at the parcel's temperature."""
    temperature = unburnt_temperature + (mv.h & parcels)                      # Scalar+[V]
    rates = prefactor * (-activation / temperature).exp()                     # Scalar+[V]
    return (mv.f & parcels) * (1 - (-dt * rates).exp())

## Diffusion

Sensible heat diffuses down the temperature gradient and fuel diffuses down its concentration gradient, with fuel diffusing faster by the inverse Lewis number. Each corner drive wedges minus a simplex gradient with its quantity direction, and an isotropic transport map built from dyads over the spatial directions maps drives to fluxes. Wedging the corner simplex gradients with the resulting fluxes and summing around each vertex assembles the cell outflow operators for heat conduction and fuel diffusion. Both operators step explicitly within the stability bound of fuel diffusion on the triangles.

In [ ]:
corners = topology.incidence[2, 0]                                            # [3] Scalar+[F <- V]
# Corner thermal and fuel drives, minus each simplex gradient wedged with its quantity, and isotropic transport maps.
heat_drives, fuel_drives = -gradients ^ mv.h, -gradients ^ mv.f               # [3] Bivector+[F], [3] Bivector+[F]
heat_sweeps = directions & heat                                               # [2] Antibivector-
fuel_sweeps = directions & fuel                                               # [2] Antibivector-
conductivity = (heat_sweeps * (heat_sweeps & Drive)).sum(axis=0)              # [] Antibivector- <- Bivector+
diffusivity = (fuel_sweeps * (fuel_sweeps & Drive)).sum(axis=0) / lewis       # [] Antibivector- <- Bivector+
# Cell heat and fuel outflow operators assembled by wedging the corner simplex gradients with the diffusive fluxes.
local = areas * (mv.h & (gradients[:, None] ^ conductivity(heat_drives[None, :])))   # [3, 3] Antipseudoscalar-[F]
conduction = (corners[:, None].adjugate() * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Antipseudoscalar-[V <- V]
local = areas * (mv.f & (gradients[:, None] ^ diffusivity(fuel_drives[None, :])))   # [3, 3] Antipseudoscalar-[F]
diffusion = (corners[:, None].adjugate() * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Antipseudoscalar-[V <- V]

## The cellular front

At each step every parcel reacts along `reaction` while sensible heat and fuel diffuse across the vertex cells, and JAX compiles the steps between consecutive frames into a single scan. The left panel shades the mixture like burning paper while the right panel colours the enthalpy `energy & parcels`. Rendering starts once the initial spark-edge noise smooths out and runs until the cellular front approaches the insulated rim. This cell takes about a minute, most of it stepping the flame.

In [ ]:
def step(parcels: Point, _) -> tuple[Point, None]:
    """Parcel field after one step of chemical reaction and explicit heat and fuel diffusion."""
    heat_rate = -(conduction * (mv.h & parcels)) / cells                      # Scalar+[V]
    fuel_rate = -(diffusion * (mv.f & parcels)) / cells                       # Scalar+[V]
    return parcels + fuel_loss(parcels) * reaction + dt * (heat_rate * heat + fuel_rate * fuel), None


@jax.jit
def advance(parcels: Point) -> Point:
    """Parcel field advanced by one frame stride."""
    return jax.lax.scan(step, parcels, None, length=scenarios.n_stride)[0]


def burning(parcels: Point, n_frames: int) -> Iterator[tuple[Point, Scalar]]:
    """Yield the mixture's parcel field and volumetric reaction rate across frames."""
    for _ in range(n_frames):
        yield parcels, fuel_loss(parcels) / dt
        parcels = advance(parcels)


n_shown = scenarios.n_frames - scenarios.n_unshown
frames = islice(burning(parcels, scenarios.n_frames), scenarios.n_unshown, None)
states = show_states(frames, render.animate, topology, vertices, scenarios.radius, energy, name="flame",
                     duration_ms=scenarios.frame_duration_ms, n_frames=n_shown)

On the left, three circular fronts expand from the sparks, pale ahead and dark behind, with bright reaction zones along their rims and red smouldering wakes where sensible heat exceeds the adiabatic rise. Where the fronts collide they merge across deep cusps, and the combined front splits into convex cells separated by dim creases that leave dark, cooler trails behind them. On the right, a blue band of enthalpy deficit runs ahead of the front where fast-diffusing fuel drains into the reaction zone before conducted heat arrives, while red enthalpy surpluses build up behind the bulges and drop along the crease trails. Because chemical reaction preserves enthalpy pointwise, the entire red-and-blue pattern arises from unequal heat and fuel diffusion.

In [ ]:
# checks
# Chemical reaction converts fuel energy into sensible heat without changing enthalpy.
assert ((energy & reaction).norm() < 1e-15).all()
# Over the early rendered frames, before the front nears the rim, total enthalpy stays constant while total sensible
# heat rises, and differential diffusion creates both an enthalpy deficit ahead of the front and a surplus behind it.
n_checked = 13 - scenarios.n_unshown
early = stack([later for later, _ in states[:n_checked]])                      # [n_checked] Antivector-[V]
initial_energy = (cells * (energy & parcels)).sites.sum()                      # [] Antipseudoscalar-
energies = (cells * (energy & early)).sites.sum()                              # [n_checked] Antipseudoscalar-
heats = (cells * (mv.h & early)).sites.sum()                                   # [n_checked] Antipseudoscalar-
assert ((energies - initial_energy).norm() < 1e-12 * initial_energy.norm()).all()
assert (((heats[1:] - heats[:-1]) / cells.sites.sum()) > 0).all()
enthalpy_change = (energy & (early[-1] - parcels)).batch()                     # [V] Scalar+
assert (enthalpy_change < -0.25).any() and (enthalpy_change > 0.2).any()